# Angine de Poitrine vs. conventional rock — a feature-based style study**Course:** DT2470 Music Informatics · **Group size:** 3## Research questionCan the unusual musical style of **Angine de Poitrine** be identified from audio features alone,and *which* measurable characteristics — particularly **rhythmic irregularity** and **microtonality** —separate it from conventional rock?## Hypotheses- **H1 (rhythm):** AdP clips show higher inter-onset-interval variability and lower beat-tracking  confidence / higher tempogram entropy than GTZAN rock.- **H2 (pitch):** AdP clips deviate more from 12-tone equal temperament (12-TET); their fine-grained  pitch-class histograms are flatter / have peaks off the semitone grid.- **H3 (classification):** A linear classifier on rhythm + pitch features separates the two classes  clearly above a majority-class baseline.## Data| Class | Source | Format | Target n ||---|---|---|---|| `adp` (positive) | 2 albums purchased on iTunes, sliced into 30 s segments | `.m4a` → `.wav` | ~100 || `rock` (negative) | GTZAN *rock* genre subset (Kaggle), already 30 s | `.wav` (22 050 Hz, mono) | 100 |## Pipeline`audio → preprocessing → feature extraction → EDA & statistics → classification (LogReg, SVM) → evaluation → interpretation`> **Note on scope:** this notebook is a *scaffold*. All code cells contain specifications only> (function name, purpose, arguments, return value, owner). Fill them in on your assigned sections.

---## Work division and working conventions| Section | Content | Owner ||---|---|---|| 0 | Setup, config, paths, helpers | **M1** (Filip) || 1 | Data acquisition, conversion, slicing, manifest | **M2** || 2 | Feature extraction (rhythm, pitch/tonal, timbre) | **M1** + **M3** (split 2.2 / 2.3) || 3 | Exploratory analysis & statistics | **M3** || 4 | Classification (splits, pipelines, models) | **M1** || 5 | Evaluation (metrics, confusion matrices, curves) | **M3** || 6 | Interpretation, ablation, error analysis | **M1** + **M3** || 7 | Confound checks | **M2** + **M1** || 8 | Export of figures/tables for the report | whoever writes that figure || 9 | Discussion & limitations | all |**Decide before you start writing code (suggested answers in parentheses):**1. Who writes 2.2 (rhythm) vs. 2.3 (pitch/tonal)? They are independent — split them.2. Sample rate and channels for *everything* (22 050 Hz, mono — matches GTZAN; resample AdP down).3. Repo layout and whether audio is tracked (it is **not** — see below).### Conventions- **Never commit audio.** The iTunes albums are purchased copyrighted material and GTZAN is large.  Put `data/` in `.gitignore`; commit only the **manifest** (`manifest.csv`) and the **feature matrix**  (`features.csv`). Everyone can then run sections 3–9 without owning the audio.- **One function per task, defined in the cell where it is specified.** Do not redefine helpers further down.- **Run order is top to bottom.** If you need something from a later section, move it earlier instead  of importing out of order.- **Cache expensive steps** (feature extraction takes minutes × 200 clips). Section 2.6 handles this.- **Naming:** `snake_case` for functions, `X` for the feature matrix, `y` for labels,  `groups` for the leakage-grouping vector (see §4.2 — this one matters).- **Randomness:** always pass `RANDOM_SEED` from §0.2. Never leave a `random_state` unset.### Expected directory layout```project/├── notebook.ipynb├── requirements.txt├── data/                     # gitignored│   ├── raw/│   │   ├── adp/              # original .m4a from iTunes│   │   └── gtzan_rock/       # original .wav from Kaggle│   ├── interim/│   │   └── adp_wav/          # full albums converted to .wav│   └── processed/│       ├── adp/              # 30 s segments│       └── rock/             # 30 s segments (copied/symlinked from raw)├── cache/│   └── features.parquet└── figures/                  # exported plots for the report```

---# 0. Setup**Owner: M1.** Everything in this section must run before any other section.

In [ ]:
# ---------------------------------------------------------------------------
# 0.1  Imports
# ---------------------------------------------------------------------------
# Group the imports so missing dependencies are obvious to whoever clones the repo.
#
#   stdlib        : os / pathlib, json, subprocess (for ffmpeg), warnings, itertools
#   numeric       : numpy, pandas, scipy.stats
#   audio         : librosa, soundfile   (librosa handles loading, resampling, all features)
#   plotting      : matplotlib.pyplot, seaborn      (pick ONE style and stick to it)
#   ml            : sklearn -> pipeline, preprocessing, linear_model, svm, model_selection,
#                   metrics, decomposition, dummy, inspection
#
# TODO(M1): mirror every import here into requirements.txt with pinned versions (see Appendix).
# NOTE: librosa is slow to import (~5 s) — that is normal, not a hang.

In [ ]:
# ---------------------------------------------------------------------------
# 0.2  Global configuration
# ---------------------------------------------------------------------------
# Single source of truth for every magic number in the notebook. Nothing below
# this cell should hard-code a sample rate, a window length or a seed.
#
#   RANDOM_SEED      = 42          reproducibility for splits, SVM, t-SNE
#   SR               = 22050       target sample rate (Hz) — GTZAN's native rate
#   MONO             = True        force single channel
#   SEGMENT_SEC      = 30.0        clip length, matches GTZAN
#   HOP_LENGTH       = 512         frame hop for all frame-wise features
#   N_FFT            = 2048        window size for STFT-based features
#   N_MFCC           = 20          number of MFCC coefficients
#   N_CHROMA_FINE    = 120         bins for the microtonal pitch histogram (= 10 cent resolution)
#   CLASS_NAMES      = {0: "rock", 1: "adp"}   -> AdP is the POSITIVE class
#
# TODO(M1): also define the directory constants (RAW_DIR, INTERIM_DIR, PROCESSED_DIR,
#           CACHE_DIR, FIG_DIR) as pathlib.Path objects relative to the notebook,
#           so the notebook runs identically on Windows and macOS.

In [ ]:
# ---------------------------------------------------------------------------
# 0.3  Reproducibility and environment helpers
# ---------------------------------------------------------------------------
# def set_seeds(seed: int = RANDOM_SEED) -> None
#     Purpose : seed numpy's global RNG and python's `random` so that any code path
#               that forgets an explicit random_state is still deterministic.
#     Args    : seed — integer
#     Returns : None (side effect only)
#
# def log_environment() -> dict
#     Purpose : record library versions + platform so the report can state exactly
#               what produced the numbers. Print it and return it.
#     Args    : none
#     Returns : dict  {"python": str, "numpy": str, "librosa": str, "sklearn": str, "platform": str}
#     Used by : Appendix A, pasted into the report's reproducibility paragraph.
#
# TODO(M1): call set_seeds() and log_environment() at the bottom of this cell.

In [ ]:
# ---------------------------------------------------------------------------
# 0.4  Directory bootstrap
# ---------------------------------------------------------------------------
# def ensure_dirs(paths: list[Path]) -> None
#     Purpose : create every directory in the expected layout if missing, so a fresh
#               clone can run end-to-end without manual mkdir.
#     Args    : paths — list of Path objects (RAW_DIR, INTERIM_DIR, ... FIG_DIR)
#     Returns : None
#     Notes   : use `p.mkdir(parents=True, exist_ok=True)`; never delete anything here.

---# 1. Data preparation**Owner: M2.** Goal of this section: turn two piles of audio in two different formats into one**manifest** — a single `DataFrame` where each row is a 30 s clip with a label and a provenance trail.Everything downstream reads the manifest, nothing downstream touches the raw folders.### Manifest schema (agree on this before writing code)| column | type | meaning ||---|---|---|| `clip_id` | str | unique, e.g. `adp_album1_track03_seg02` || `path` | str | path to the 30 s `.wav` on disk || `label` | int | `1` = adp, `0` = rock || `label_name` | str | `"adp"` / `"rock"` || `source_track` | str | the parent track the segment was cut from — **needed for §4.2** || `source_album` | str | album / GTZAN file group — **needed for §7.1** || `segment_idx` | int | position of the segment within the parent track || `duration_sec` | float | measured, not assumed || `sr` | int | measured sample rate before resampling |> ⚠️ `source_track` and `source_album` are the two most important columns in the whole project.> Without them you cannot prevent data leakage in §4 and the classification results will be> meaningless (and a reviewer will spot it immediately).

In [ ]:
# ---------------------------------------------------------------------------
# 1.1  Convert the purchased iTunes albums from .m4a (AAC) to .wav
# ---------------------------------------------------------------------------
# def convert_m4a_to_wav(src_dir: Path, dst_dir: Path, sr: int = SR, mono: bool = MONO) -> list[Path]
#     Purpose : decode every .m4a in src_dir to 16-bit PCM .wav in dst_dir, resampled to `sr`
#               and downmixed to mono, so AdP and GTZAN are byte-level comparable.
#     Args    : src_dir — folder with the original purchased files
#               dst_dir — output folder (INTERIM_DIR / "adp_wav")
#               sr, mono — target format
#     Returns : list of written .wav paths
#     How     : EITHER shell out to ffmpeg
#                   ffmpeg -i in.m4a -ac 1 -ar 22050 -sample_fmt s16 out.wav
#               via subprocess.run(..., check=True),
#               OR use librosa.load(path, sr=SR, mono=True) + soundfile.write().
#               ffmpeg is faster and handles DRM-free AAC fine; librosa avoids an external dependency.
#               -> M2 decides and documents the choice here.
#     Notes   : * iTunes purchases are DRM-free .m4a; if a file refuses to decode, it is protected
#                 and must be re-downloaded or replaced.
#               * Skip files that already exist in dst_dir so the cell is cheap to re-run.
#               * Keep the original track title in the filename — it becomes `source_track`.

In [ ]:
# ---------------------------------------------------------------------------
# 1.2  Slice full tracks into fixed-length 30 s segments
# ---------------------------------------------------------------------------
# def slice_track(wav_path: Path, dst_dir: Path, segment_sec: float = SEGMENT_SEC,
#                 drop_last_partial: bool = True, skip_lead_sec: float = 0.0) -> list[dict]
#     Purpose : cut one long .wav into consecutive non-overlapping 30 s segments.
#     Args    : wav_path          — the full track
#               dst_dir           — PROCESSED_DIR / "adp"
#               segment_sec       — clip length
#               drop_last_partial — discard a final segment shorter than segment_sec
#                                   (a 7 s tail would bias duration-sensitive features)
#               skip_lead_sec     — optionally drop the first N seconds (intros / silence)
#     Returns : list of manifest row dicts (see schema above), one per written segment
#     Notes   : * NON-OVERLAPPING. Overlapping windows inflate n but duplicate information and
#                 leak across the train/test split.
#               * Consider dropping near-silent segments — see 1.5.
#               * ~100 clips from 2 albums means roughly 50 min of audio: check that your albums
#                 actually give you that, and record the real number in the report.
#
# def slice_all(src_dir: Path, dst_dir: Path) -> pd.DataFrame
#     Purpose : run slice_track over every converted album track.
#     Returns : DataFrame of manifest rows for the `adp` class.

In [ ]:
# ---------------------------------------------------------------------------
# 1.3  Index the GTZAN rock subset
# ---------------------------------------------------------------------------
# def index_gtzan_rock(src_dir: Path, n: int = 100, seed: int = RANDOM_SEED) -> pd.DataFrame
#     Purpose : select n GTZAN rock clips and build manifest rows for them. They are already
#               30 s / 22 050 Hz / mono, so no cutting is needed — only indexing and validation.
#     Args    : src_dir — folder with rock.00000.wav ... rock.00099.wav
#               n       — how many to keep (match the adp count for a balanced set)
#               seed    — for the random subset if more than n are available
#     Returns : DataFrame of manifest rows, label = 0
#     Notes   : * GTZAN's rock genre has exactly 100 clips, so n=100 usually means "take all".
#                 If AdP yields fewer than 100 segments, subsample rock to match and say so.
#               * `source_track` for GTZAN = the filename (each file is its own track).
#                 `source_album` = the filename too, unless you know the artist groupings.
#               * KNOWN ISSUE: GTZAN contains exact duplicates, mislabelled clips and distortion
#                 (Sturm, 2013, "The GTZAN dataset: its contents, its faults..."). Mention this in
#                 the limitations section and, if time allows, drop known-bad indices.
#               * One GTZAN file (jazz.00054) is corrupt — not in rock, but be aware.

In [ ]:
# ---------------------------------------------------------------------------
# 1.4  Build and persist the combined manifest
# ---------------------------------------------------------------------------
# def build_manifest(adp_df: pd.DataFrame, rock_df: pd.DataFrame, out_csv: Path) -> pd.DataFrame
#     Purpose : concatenate both classes, enforce the schema, assert uniqueness of clip_id,
#               write to CSV and return it.
#     Args    : adp_df, rock_df — outputs of 1.2 and 1.3
#               out_csv         — data/manifest.csv  (THIS file gets committed to git)
#     Returns : the combined manifest, sorted by clip_id
#     Asserts : no duplicate clip_id; no missing paths; label ∈ {0,1}; every path exists on disk.
#
# def load_manifest(path: Path) -> pd.DataFrame
#     Purpose : the entry point for M1 and M3 — lets them start at §2 without regenerating audio.
#     Returns : manifest DataFrame

In [ ]:
# ---------------------------------------------------------------------------
# 1.5  Dataset validation and quality report
# ---------------------------------------------------------------------------
# def validate_dataset(manifest: pd.DataFrame) -> pd.DataFrame
#     Purpose : catch format mismatches BEFORE two days of feature extraction.
#     Checks  : - every file opens and has duration ≈ SEGMENT_SEC (tolerance ±0.1 s)
#               - sample rate == SR for all files after conversion
#               - channels == 1
#               - no all-silent or near-silent clips (RMS below a threshold)
#               - no clipping (fraction of samples at |x| >= 0.999)
#               - class balance: print value_counts of label
#               - clips per source_track: print the distribution (this previews the leakage risk)
#     Returns : a per-clip diagnostics DataFrame; print a summary and raise/warn on failures.
#
# def plot_dataset_overview(manifest, diagnostics) -> None
#     Purpose : one figure for the report's "Data" section — class counts, clips per album/track,
#               RMS-loudness distribution per class.
#     Returns : None (saves to FIG_DIR via §8.1)

In [ ]:
# ---------------------------------------------------------------------------
# 1.6  Canonical audio loader (used by EVERY feature function)
# ---------------------------------------------------------------------------
# def load_audio(path: str | Path, sr: int = SR, mono: bool = MONO,
#                normalize: bool = True) -> tuple[np.ndarray, int]
#     Purpose : the single place where audio enters the analysis. Guarantees that an AdP clip and
#               a GTZAN clip are presented to the feature extractors on identical terms.
#     Args    : path      — .wav path from the manifest
#               sr, mono  — target format (resamples if needed)
#               normalize — peak- or RMS-normalise the waveform
#     Returns : (y, sr) — float32 waveform in [-1, 1], and the actual sample rate
#     Notes   : ⚠️ LOUDNESS IS A CONFOUND. iTunes masters are mastered loud; GTZAN clips are
#               ripped from 1990s/2000s CDs at varying levels. If you do not normalise, the
#               classifier can learn "which decade was this mastered in" instead of "is this AdP".
#               RECOMMENDATION: RMS-normalise to a fixed target (e.g. -20 dBFS) rather than peak-
#               normalise, and record in the report which you used. See §7.3 for the control test.

### M2 checklist before handing over to §2- [ ] All AdP tracks converted, none failed to decode- [ ] Segment count per class recorded (write the real numbers into the report, not "~100")- [ ] `validate_dataset` passes with zero errors- [ ] `manifest.csv` committed; `data/` in `.gitignore`- [ ] `source_track` and `source_album` populated for **both** classes- [ ] Decision documented: RMS-normalisation target, whether silent clips were dropped

---# 2. Feature extraction**Owners: M1 (2.2 rhythm) and M3 (2.3 tonal), jointly 2.5–2.7.**Every extractor follows the same contract:```extract_<group>_features(y: np.ndarray, sr: int) -> dict[str, float]```A **flat dict of scalars**, keys prefixed by the group (`rhy_`, `pit_`, `tim_`). One clip in,one row of numbers out. No DataFrames inside the extractors, no plotting inside the extractors.**Why scalars:** librosa returns time series (one value per frame). The classifier needs a fixed-lengthvector, so each time series must be summarised — typically `mean`, `std`, and for the irregularityhypothesis also `iqr` / `entropy`. **The summary statistic is itself a modelling choice:** the *std*of the inter-onset intervals is literally the rhythmic-irregularity hypothesis, so do not just take means.

In [ ]:
# ---------------------------------------------------------------------------
# 2.1  Shared frame-level primitives (compute once, reuse across extractors)
# ---------------------------------------------------------------------------
# def compute_primitives(y: np.ndarray, sr: int) -> dict
#     Purpose : STFT-derived representations are expensive; several feature groups need the same
#               ones. Compute them once per clip and pass the dict down.
#     Returns : dict with keys such as
#               "S"         — magnitude spectrogram, np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
#               "onset_env" — librosa.onset.onset_strength(...)
#               "harmonic", "percussive" — librosa.effects.hpss(y)   (harmonic -> pitch, percussive -> rhythm)
#     Notes   : HPSS roughly doubles extraction time but markedly improves both chroma and onset
#               estimates. Decide as a group whether to use it and apply the choice to BOTH classes.

In [ ]:
# ---------------------------------------------------------------------------
# 2.2  Rhythm features  —  tests H1 (rhythmic irregularity)       OWNER: M1
# ---------------------------------------------------------------------------
# def extract_rhythm_features(y: np.ndarray, sr: int, prim: dict | None = None) -> dict[str, float]
#     Purpose : quantify tempo, pulse clarity and — most importantly — how *regular* the
#               rhythmic events are.
#     Args    : y, sr — waveform from load_audio(); prim — optional output of compute_primitives()
#     Returns : flat dict, keys prefixed "rhy_"
#
#     Suggested features (librosa functions in brackets):
#       rhy_tempo_global          global tempo estimate            [librosa.beat.beat_track]
#       rhy_tempo_std             std of a dynamic tempo curve     [librosa.feature.tempo(aggregate=None)]
#                                 -> high value = tempo drifts, a direct irregularity marker
#       rhy_beat_count            number of tracked beats in 30 s
#       rhy_pulse_clarity         max of the normalised autocorr. tempogram
#                                 -> low value = no stable pulse; expected to be LOW for AdP
#       rhy_tempogram_entropy     Shannon entropy of the mean tempogram over lags
#                                 -> high value = no dominant periodicity
#       rhy_onset_rate            onsets per second                [librosa.onset.onset_detect]
#       rhy_ioi_mean              mean inter-onset interval
#       rhy_ioi_std               std of IOIs          <-- the core H1 feature
#       rhy_ioi_cv                coefficient of variation = std/mean  <-- tempo-invariant version,
#                                 PREFER THIS: raw std scales with tempo, CV does not
#       rhy_ioi_entropy           entropy of the IOI histogram
#       rhy_beat_sync_dev         mean |onset time - nearest tracked beat time|, in seconds or
#                                 as a fraction of the beat period -> how far events sit off-grid
#       rhy_onset_env_mean/std    strength and variability of the onset envelope
#
#     Notes   : * Beat trackers assume a steady pulse. On AdP they may fail — that failure is
#                 itself a signal, but make sure the function degrades gracefully (return np.nan
#                 rather than raising) and that §2.7 reports how often it happens.
#               * Report tempo-invariant versions wherever possible, otherwise the classifier may
#                 just be learning "AdP is slower/faster".

In [ ]:
# ---------------------------------------------------------------------------
# 2.3  Pitch / tonal features  —  tests H2 (microtonality)        OWNER: M3
# ---------------------------------------------------------------------------
# def extract_tonal_features(y: np.ndarray, sr: int, prim: dict | None = None) -> dict[str, float]
#     Purpose : quantify the pitch content and, specifically, how much it departs from
#               12-tone equal temperament.
#     Returns : flat dict, keys prefixed "pit_"
#
#     Standard 12-bin chroma:
#       pit_chroma_mean_00..11    mean energy per pitch class      [librosa.feature.chroma_cqt]
#       pit_chroma_std_00..11     variability per pitch class
#       pit_chroma_entropy        entropy of the mean chroma vector -> high = no tonal centre
#       pit_tonal_centroid_*      6 tonnetz dimensions             [librosa.feature.tonnetz]
#       pit_key_strength          max of a key-profile correlation (Krumhansl) -> weak key = low
#
#     Microtonality — THE interesting part, do not skip it:
#       pit_fine_hist_entropy     entropy of a 120-bin (10-cent) pitch-class histogram
#                                 [librosa.feature.chroma_cqt(bins_per_octave=120, n_chroma=120)]
#                                 -> a 12-TET piece concentrates energy in 12 of the 120 bins;
#                                    a microtonal piece spreads it out
#       pit_tet_deviation_mean    mean absolute distance (in cents) from each detected f0 to the
#                                 nearest 12-TET semitone, range 0..50 cents
#                                 -> expected HIGHER for AdP; this is the single most direct test of H2
#       pit_tet_deviation_std     variability of that deviation
#       pit_tuning_offset         global tuning estimate            [librosa.estimate_tuning]
#                                 -> separates "tuned a quarter-tone flat overall" (a constant offset,
#                                    NOT microtonality) from "uses intervals outside 12-TET"
#                                    ⚠️ IMPORTANT: compute pit_tet_deviation AFTER removing this
#                                    global offset, otherwise a band that simply tuned down looks microtonal.
#       pit_f0_median / _iqr      pitch range and spread            [librosa.pyin or librosa.yin]
#       pit_voiced_fraction       fraction of frames with a confident f0 -> how pitched the music is
#       pit_interval_hist_*       histogram of successive f0 intervals in cents; count the fraction
#                                 falling within ±X cents of a 12-TET interval
#
#     Notes   : * pyin is SLOW (tens of seconds per clip). Budget for it, or use yin with a
#                 restricted fmin/fmax and document the trade-off.
#               * Choose fmin/fmax deliberately (e.g. 65–2000 Hz) and use the same values for both classes.

In [ ]:
# ---------------------------------------------------------------------------
# 2.4  Timbre / spectral features (MFCC baseline)
# ---------------------------------------------------------------------------
# def extract_timbre_features(y: np.ndarray, sr: int, prim: dict | None = None) -> dict[str, float]
#     Purpose : the conventional MIR baseline. Serves two roles: (a) a strong comparison for
#               rhythm+pitch, (b) a CONFOUND DETECTOR — see §7.2.
#     Returns : flat dict, keys prefixed "tim_"
#
#       tim_mfcc_{i}_mean / _std         i = 1..N_MFCC            [librosa.feature.mfcc]
#       tim_mfcc_delta_{i}_mean / _std   first-order deltas       [librosa.feature.delta]
#       tim_centroid_mean / _std         spectral centroid
#       tim_bandwidth_mean / _std        spectral bandwidth
#       tim_rolloff_mean / _std          spectral rolloff (85 %)
#       tim_flatness_mean / _std         spectral flatness -> noisiness
#       tim_zcr_mean / _std              zero-crossing rate
#       tim_rms_mean / _std              loudness envelope (keep it even after normalisation —
#                                        dynamic range still differs between masterings)
#
#     ⚠️ Read §7.2 before interpreting any MFCC result. If MFCCs alone give ~100 % accuracy, the
#        model has very likely learned recording/production differences rather than musical style.

In [ ]:
# ---------------------------------------------------------------------------
# 2.5  Dispatcher — all features for one clip
# ---------------------------------------------------------------------------
# def extract_all_features(path: str | Path, groups: tuple[str, ...] = ("rhythm", "tonal", "timbre")
#                          ) -> dict[str, float]
#     Purpose : load one clip, run the requested extractors, merge their dicts into one row.
#     Args    : path   — .wav path from the manifest
#               groups — which feature families to compute (lets §7.2 request subsets cheaply)
#     Returns : one flat dict; keys must be identical across all clips
#     Notes   : * Wrap each extractor in try/except: a single failing clip must not kill a 20-minute run.
#                 On failure, fill that group's keys with np.nan and log the clip_id.
#               * Call compute_primitives() once and pass it to all three extractors.

In [ ]:
# ---------------------------------------------------------------------------
# 2.6  Build the feature matrix (with caching)
# ---------------------------------------------------------------------------
# def build_feature_matrix(manifest: pd.DataFrame, cache_path: Path,
#                          force: bool = False, n_jobs: int = 1) -> pd.DataFrame
#     Purpose : run extract_all_features over every clip and return one wide DataFrame.
#     Args    : manifest   — from §1.4
#               cache_path — cache/features.parquet (or .csv)
#               force      — ignore the cache and recompute
#               n_jobs     — joblib.Parallel workers; librosa releases the GIL so this helps
#     Returns : DataFrame indexed by clip_id, with all feature columns PLUS the carried-over
#               manifest columns label, source_track, source_album  <-- do not drop these
#     Notes   : * Show a progress bar (tqdm). 200 clips × pyin is not instant.
#               * Save to the cache on success so §3–§9 reload in a second.
#               * COMMIT THE CACHED FEATURE FILE to git — it is small, and it lets group members
#                 without the audio reproduce every result from §3 onwards.
#
# def load_features(cache_path: Path) -> pd.DataFrame
#     Purpose : entry point for M3 — start the analysis without owning the audio.

In [ ]:
# ---------------------------------------------------------------------------
# 2.7  Feature sanity checks
# ---------------------------------------------------------------------------
# def audit_features(X: pd.DataFrame) -> pd.DataFrame
#     Purpose : find problems that would silently wreck the classifier.
#     Checks  : - NaN / inf count per column, and per class (a feature that is NaN only for AdP
#                 is a perfect leak: the imputer value becomes a class label)
#               - zero-variance columns (drop them)
#               - near-duplicate columns (|r| > 0.99)
#               - extreme outliers (|z| > 6) — inspect the clip, do not auto-delete
#               - how often the beat tracker / pitch tracker failed, per class
#     Returns : an audit DataFrame; print a short verdict.
#
# def handle_missing(X: pd.DataFrame, strategy: str = "median") -> pd.DataFrame
#     Purpose : impute remaining NaNs.
#     ⚠️ Imputation must be fitted INSIDE the cross-validation pipeline (§4.3), not here on the
#        full dataset — otherwise test-set statistics leak into training. If you impute here,
#        say so in the report and accept the (small) optimism.

---# 3. Exploratory analysis and statistics**Owner: M3.**This section answers *"how do the features differ between the groups?"* — the first half of theproject aim — and it is reported **independently of the classifier**. Do this before modelling:if no feature separates the classes, a 99 % accuracy in §4 means something has gone wrong.

In [ ]:
# ---------------------------------------------------------------------------
# 3.1  Descriptive summary table
# ---------------------------------------------------------------------------
# def summarize_by_class(X: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame
#     Purpose : per feature, per class: n, mean, std, median, IQR.
#     Returns : tidy DataFrame, one row per (feature, class) or one row per feature with
#               adp_/rock_ prefixed columns — pick the shape that exports best to the report table.
#     Output  : this is Table 1 of the report. Export it via §8.2.

In [ ]:
# ---------------------------------------------------------------------------
# 3.2  Distribution plots
# ---------------------------------------------------------------------------
# def plot_feature_distributions(X: pd.DataFrame, features: list[str], kind: str = "violin") -> Figure
#     Purpose : visual class comparison for a handful of hand-picked features.
#     Args    : features — SHORTLIST, not all 150. Suggested: rhy_ioi_cv, rhy_pulse_clarity,
#                          rhy_tempo_std, pit_tet_deviation_mean, pit_fine_hist_entropy,
#                          pit_chroma_entropy, tim_mfcc_1_mean
#               kind     — "violin" | "box" | "hist"
#     Returns : matplotlib Figure (so §8.1 can save it)
#     Notes   : overlay individual points (stripplot) — with n≈100 per class the reader should see
#               the actual spread, not just a smoothed shape.
#
# def plot_hypothesis_panel(X) -> Figure
#     Purpose : ONE figure with 2 panels, H1 (rhy_ioi_cv) and H2 (pit_tet_deviation_mean).
#               This is the figure the report is built around — make it the good one.

In [ ]:
# ---------------------------------------------------------------------------
# 3.3  Statistical tests and effect sizes
# ---------------------------------------------------------------------------
# def compare_groups(X: pd.DataFrame, feature_cols: list[str], alpha: float = 0.05) -> pd.DataFrame
#     Purpose : test each feature for a difference between adp and rock.
#     Method  : Mann-Whitney U (non-parametric — audio features are rarely normal) or Welch's t-test.
#               Report an EFFECT SIZE alongside p: Cliff's delta (for Mann-Whitney) or Cohen's d.
#               Apply a multiple-comparison correction across all tested features
#               (Benjamini-Hochberg FDR; `statsmodels.stats.multitest.multipletests`).
#     Returns : DataFrame [feature, stat, p_raw, p_adjusted, effect_size, significant]
#               sorted by |effect_size| descending.
#     ⚠️ With ~150 features and alpha=0.05 you expect ~7 false positives by chance. The correction
#        is not optional, and neither is reporting effect sizes — with n=200 almost everything
#        becomes "significant" while being practically irrelevant.
#     ⚠️ A second, deeper caveat: the 30 s segments from one album are NOT independent observations,
#        so the nominal p-values are optimistic. State this in the limitations.

In [ ]:
# ---------------------------------------------------------------------------
# 3.4  Redundancy and correlation structure
# ---------------------------------------------------------------------------
# def plot_correlation_heatmap(X, feature_cols, cluster: bool = True) -> Figure
#     Purpose : show which features are redundant (MFCCs and spectral features are highly collinear).
#               Motivates regularisation in §4 and explains why logistic-regression coefficients
#               must be read with care in §6.1.
#     Returns : Figure
#
# def drop_collinear(X, feature_cols, threshold: float = 0.95) -> list[str]
#     Purpose : optional pruning — return the feature list with one of each |r|>threshold pair removed.
#     Returns : reduced feature-name list

In [ ]:
# ---------------------------------------------------------------------------
# 3.5  Low-dimensional projection
# ---------------------------------------------------------------------------
# def plot_projection(X, y, method: str = "pca", groups=None) -> Figure
#     Purpose : do the classes form separate clouds at all? A quick visual feasibility check.
#     Args    : method — "pca" (linear, interpretable, report the explained variance ratio)
#                      | "tsne" (non-linear; set random_state and state the perplexity)
#               groups — optional: colour by source_album instead of class. DO THIS TOO.
#                        If the AdP points cluster by *album* rather than forming one cloud,
#                        you are partly measuring production, not style (see §7).
#     Returns : Figure
#     ⚠️ Fit the scaler on the whole set here — this is exploration, not evaluation.
#        Never reuse this fitted scaler in §4.

---# 4. Classification**Owner: M1.**### The single most important design decision in this projectYou have ~100 AdP clips drawn from **2 albums**, i.e. maybe 20 tracks. Segments from the same trackshare tempo, key, instrumentation, mix and mastering. If a random train/test split puts`adp_album1_track03_seg01` in train and `adp_album1_track03_seg02` in test, the model can recognisethe *track*, not the *style* — and you will report 99 % accuracy that means nothing.**Therefore: split by `source_track`, not by clip.** Use `GroupShuffleSplit` / `GroupKFold` with`groups = source_track`. Report the grouped result as the headline number. If you also run a naiverandom split, present it only as a demonstration of the leakage effect — that contrast is a genuinelygood thing to include in the report.A stricter version (§7.1): hold out an entire *album*. Hardest and most honest.

In [ ]:
# ---------------------------------------------------------------------------
# 4.1  Assemble X, y, groups
# ---------------------------------------------------------------------------
# def prepare_xy(features_df: pd.DataFrame, feature_cols: list[str] | None = None
#                ) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]
#     Purpose : separate the design matrix from the labels and the grouping vector.
#     Args    : features_df  — output of §2.6
#               feature_cols — None = all numeric feature columns; else a subset (used by §7.2)
#     Returns : (X, y, groups) where
#                 X      — DataFrame of features only (no label, no ids)
#                 y      — int array, 1 = adp, 0 = rock
#                 groups — array of source_track strings, SAME LENGTH as y
#     Asserts : len(X) == len(y) == len(groups); no id/label columns left in X.

In [ ]:
# ---------------------------------------------------------------------------
# 4.2  Leakage-safe splitting
# ---------------------------------------------------------------------------
# def make_holdout(X, y, groups, test_size: float = 0.25, seed: int = RANDOM_SEED)
#         -> tuple[np.ndarray, np.ndarray]
#     Purpose : one grouped train/test split for the final held-out evaluation.
#     Method  : sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size, random_state=seed)
#     Returns : (train_idx, test_idx)
#     Asserts : the intersection of source_track between train and test is EMPTY — assert it,
#               do not assume it. Print the resulting class balance in each split.
#
# def make_cv(n_splits: int = 5) -> BaseCrossValidator
#     Purpose : the CV object used everywhere in §4.5–4.6.
#     Method  : StratifiedGroupKFold(n_splits, shuffle=True, random_state=RANDOM_SEED)
#               -> keeps class balance AND respects groups. Plain GroupKFold is the fallback.
#     Returns : the splitter
#     Notes   : with only ~20 AdP tracks, 5 folds means ~4 tracks held out per fold. Expect
#               HIGH VARIANCE between folds — always report mean ± std, never a single number.
#               Consider repeating the CV with several seeds.

In [ ]:
# ---------------------------------------------------------------------------
# 4.3  Model pipelines
# ---------------------------------------------------------------------------
# def build_pipeline(model_name: str, **model_kwargs) -> Pipeline
#     Purpose : wrap imputation + scaling + estimator so that preprocessing is refitted inside
#               every CV fold. This is what prevents preprocessing leakage.
#     Args    : model_name — "logreg" | "svm_linear" | "svm_rbf" | "dummy" (| "rf" if you want a
#                            non-linear reference)
#     Returns : sklearn Pipeline:
#                 SimpleImputer(strategy="median")
#                 -> StandardScaler()
#                 -> estimator
#     Estimators:
#         logreg      LogisticRegression(penalty="l2", C=..., max_iter=5000,
#                                        class_weight="balanced", random_state=SEED)
#         svm_linear  SVC(kernel="linear", C=..., class_weight="balanced", random_state=SEED)
#         svm_rbf     SVC(kernel="rbf", C=..., gamma=..., probability=True, ...)
#     Notes   : * SCALING IS MANDATORY for both LogReg and SVM — MFCCs and tempo differ by orders
#                 of magnitude and an unscaled SVM will simply be wrong.
#               * Set probability=True only if §5.3 needs ROC curves; it makes SVC much slower.
#                 Alternative: use decision_function() for the ROC instead.

In [ ]:
# ---------------------------------------------------------------------------
# 4.4  Baselines (run these FIRST)
# ---------------------------------------------------------------------------
# def run_baselines(X, y, groups, cv) -> pd.DataFrame
#     Purpose : establish the floor. Every later number is meaningless without it.
#     Baselines:
#         - DummyClassifier(strategy="most_frequent")  -> ~50 % on a balanced set
#         - DummyClassifier(strategy="stratified")
#         - LogReg on ONE feature only (e.g. rhy_ioi_cv) -> how far does a single feature get you?
#     Returns : results DataFrame in the same shape as §4.5 so they can be concatenated.

In [ ]:
# ---------------------------------------------------------------------------
# 4.5  Cross-validated model comparison
# ---------------------------------------------------------------------------
# def cross_validate_model(pipeline, X, y, groups, cv, scoring: list[str]) -> dict
#     Purpose : per-fold scores for one model.
#     Args    : scoring — ["accuracy", "precision", "recall", "f1", "roc_auc"]
#     Method  : sklearn.model_selection.cross_validate(..., groups=groups, cv=cv,
#                                                      return_train_score=True)
#     Returns : dict of per-fold arrays + their mean/std
#     Notes   : return_train_score=True is worth it — a large train/test gap reveals overfitting
#               that the test score alone hides.
#
# def compare_models(X, y, groups, cv, model_names: list[str]) -> pd.DataFrame
#     Purpose : run cross_validate_model for every model and stack the results.
#     Returns : DataFrame [model, metric, mean, std] -> Table 2 of the report.

In [ ]:
# ---------------------------------------------------------------------------
# 4.6  Hyperparameter search
# ---------------------------------------------------------------------------
# def tune_model(model_name: str, X, y, groups, cv, param_grid: dict) -> GridSearchCV
#     Purpose : pick C (and gamma) without leaking the test set.
#     Grids   : logreg / svm_linear : C ∈ [0.001, 0.01, 0.1, 1, 10, 100]
#               svm_rbf             : C as above × gamma ∈ ["scale", 0.001, 0.01, 0.1]
#     Method  : GridSearchCV(pipeline, param_grid, cv=cv, scoring="f1", refit=True)
#               and pass `groups=groups` to .fit().
#     Returns : the fitted GridSearchCV
#     ⚠️ Tune on the TRAIN portion only. Tuning on all data and then reporting CV scores from the
#        same data is the second most common leak after the grouping issue.
#     ⚠️ With ~20 groups, nested CV is statistically cleaner but probably too noisy here. Simpler
#        and defensible: tune with grouped CV on train, evaluate once on the held-out groups. Say
#        which you did.

In [ ]:
# ---------------------------------------------------------------------------
# 4.7  Final fit and held-out prediction
# ---------------------------------------------------------------------------
# def fit_final(best_pipeline, X_train, y_train) -> Pipeline
#     Purpose : refit the selected configuration on the full training portion.
#     Returns : fitted pipeline
#
# def predict_holdout(fitted, X_test) -> tuple[np.ndarray, np.ndarray]
#     Purpose : predictions for the untouched held-out tracks.
#     Returns : (y_pred, y_score) where y_score is predict_proba[:,1] or decision_function
#     ⚠️ Run this cell ONCE, at the very end. Every time you look at the test set and then change
#        something, it quietly stops being a test set.

---# 5. Evaluation**Owner: M3.** Metrics required by the project description: accuracy, precision, recall, F1,confusion matrix. Positive class = `adp`, so *recall* reads as "what fraction of AdP clips did we catch".

In [ ]:
# ---------------------------------------------------------------------------
# 5.1  Metric computation
# ---------------------------------------------------------------------------
# def compute_metrics(y_true, y_pred, y_score=None, pos_label: int = 1) -> dict[str, float]
#     Purpose : one dict of all headline metrics for a single evaluation.
#     Returns : {"accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy"}
#     Notes   : state explicitly in the report that metrics are computed for the positive class
#               `adp` (sklearn's `average` argument: use "binary" with pos_label=1).
#
# def metrics_table(results: dict[str, dict]) -> pd.DataFrame
#     Purpose : stack metrics for several models into the report's results table.
#     Returns : DataFrame, models as rows, metrics as columns, values formatted "mean ± std".

In [ ]:
# ---------------------------------------------------------------------------
# 5.2  Confusion matrices
# ---------------------------------------------------------------------------
# def plot_confusion(y_true, y_pred, labels=("rock", "adp"), normalize: str | None = None) -> Figure
#     Purpose : the confusion matrix for the report.
#     Args    : normalize — None (counts) | "true" (row-wise rates). Show BOTH: counts give the
#                           reader n, rates make the classes comparable.
#     Returns : Figure
#
# def plot_cv_confusion(pipeline, X, y, groups, cv) -> Figure
#     Purpose : aggregate confusion matrix over all CV folds via cross_val_predict(..., groups=groups).
#               More stable than a single split with this little data — preferable as the main figure.

In [ ]:
# ---------------------------------------------------------------------------
# 5.3  Threshold-independent curves
# ---------------------------------------------------------------------------
# def plot_roc_pr(y_true, y_score) -> Figure
#     Purpose : ROC and precision-recall curves side by side, with AUC in the legend.
#     Returns : Figure
#     Notes   : for a balanced dataset ROC is fine; include PR anyway, it is more informative if
#               your final class counts end up uneven.

In [ ]:
# ---------------------------------------------------------------------------
# 5.4  Per-group error breakdown
# ---------------------------------------------------------------------------
# def errors_by_group(manifest, y_true, y_pred, group_col: str = "source_track") -> pd.DataFrame
#     Purpose : accuracy per source track / album. Reveals whether errors concentrate in one or
#               two tracks (an outlier problem) or are spread evenly (a genuine style boundary).
#     Returns : DataFrame [group, n_clips, n_correct, accuracy] sorted ascending by accuracy.
#     Output  : the worst rows feed the error analysis in §6.4.

---# 6. Interpretation**Owners: M1 + M3.** The project aim is *"which measurable characteristics distinguish AdP"* —so this section, not §5, is where the research question is actually answered. Budget real time for it.

In [ ]:
# ---------------------------------------------------------------------------
# 6.1  Linear model coefficients
# ---------------------------------------------------------------------------
# def top_coefficients(fitted_pipeline, feature_names, k: int = 20) -> pd.DataFrame
#     Purpose : largest positive/negative weights of the fitted LogReg or linear SVM.
#     Returns : DataFrame [feature, coefficient, abs_coefficient], sorted
#     Notes   : valid only because features were standardised — say so.
#     ⚠️ With collinear features (§3.4) the weight is split arbitrarily between correlated
#        columns. A small coefficient does NOT mean the feature is unimportant. Use §6.2 as a
#        cross-check and do not over-claim from coefficients alone.
#
# def plot_coefficients(coef_df, k: int = 20) -> Figure

In [ ]:
# ---------------------------------------------------------------------------
# 6.2  Permutation importance
# ---------------------------------------------------------------------------
# def permutation_importances(fitted_pipeline, X_test, y_test, n_repeats: int = 30) -> pd.DataFrame
#     Purpose : model-agnostic importance — how much does the score drop when one feature is shuffled?
#               Works for the RBF SVM, where coefficients do not exist.
#     Method  : sklearn.inspection.permutation_importance(..., random_state=RANDOM_SEED)
#     Returns : DataFrame [feature, importance_mean, importance_std], sorted
#     ⚠️ Also distorted by collinearity (shuffling one of two duplicated features changes little).
#        Prefer importance over FEATURE GROUPS (§6.3) as the headline claim.

In [ ]:
# ---------------------------------------------------------------------------
# 6.3  Feature-group ablation   <-- the cleanest answer to the research question
# ---------------------------------------------------------------------------
# def ablation_study(features_df, y, groups, cv, group_defs: dict[str, list[str]]) -> pd.DataFrame
#     Purpose : retrain the same model on different feature subsets and compare CV F1.
#               This directly tests "is rhythm enough?", "is pitch enough?", "do MFCCs dominate?".
#     Args    : group_defs — e.g.
#                 {"rhythm_only":   [cols starting with "rhy_"],
#                  "tonal_only":    [cols starting with "pit_"],
#                  "timbre_only":   [cols starting with "tim_"],
#                  "rhythm+tonal":  rhy_ + pit_,            <-- the hypothesis-driven set
#                  "all":           everything,
#                  "irregularity_core": [rhy_ioi_cv, rhy_pulse_clarity, rhy_tempo_std,
#                                        pit_tet_deviation_mean, pit_fine_hist_entropy]}
#     Returns : DataFrame [feature_set, n_features, f1_mean, f1_std, accuracy_mean, ...]
#     Output  : Table 3 and probably the most quotable result in the report.

In [ ]:
# ---------------------------------------------------------------------------
# 6.4  Error analysis (listen to the mistakes)
# ---------------------------------------------------------------------------
# def list_misclassified(manifest, y_true, y_pred, y_score, n: int = 10) -> pd.DataFrame
#     Purpose : the most confidently wrong clips, with their paths.
#     Returns : DataFrame [clip_id, path, true, pred, score], sorted by confidence of the error
#
# def play_clip(path) -> IPython.display.Audio
#     Purpose : embed an audio player so you can actually LISTEN to the failures.
#               This is music informatics: the qualitative listening pass belongs in the report.
#     Returns : Audio widget
#     Notes   : write 2–3 sentences per misclassified clip about what you hear. Typical findings:
#               the AdP clip is an ambient intro with no onsets; the GTZAN clip is prog rock in 7/8.

---# 7. Confound checks**Owners: M2 + M1.** *This section is not in the original project description — add it.*Your two classes differ in more than musical style: different artists, different studios, differentmastering eras, different encoders (AAC→WAV vs. GTZAN's heavily compressed 22 kHz MP3 origins).A classifier can reach high accuracy on any of those. If you do not test for it, a grader will ask,and you will not have an answer.The point is not that the project is flawed — it is that **showing you tested for the confound andreporting what you found is worth more marks than a high accuracy number.** Expect the honest resultto be "style features carry real signal, but production differences contribute too", and say so.

In [ ]:
# ---------------------------------------------------------------------------
# 7.1  Album-level hold-out (the hardest generalisation test)
# ---------------------------------------------------------------------------
# def leave_one_album_out(features_df, y, album_col: str = "source_album") -> pd.DataFrame
#     Purpose : train on album 1, test on album 2, and vice versa.
#               If performance collapses, the model learned album-specific production, not style.
#     Method  : LeaveOneGroupOut with groups = source_album (restricted to the adp side; pair each
#               album with a disjoint random half of the rock clips).
#     Returns : DataFrame [held_out_album, n_test, accuracy, f1]
#     Notes   : with only 2 albums this is 2 folds — noisy, but highly informative. Report it.

In [ ]:
# ---------------------------------------------------------------------------
# 7.2  Style features vs. timbre/production features
# ---------------------------------------------------------------------------
# def confound_contrast(features_df, y, groups, cv) -> pd.DataFrame
#     Purpose : compare three models:
#                 A) timbre/MFCC only          -> mostly production + instrumentation
#                 B) rhythm + tonal only       -> the hypothesised style features
#                 C) all features
#     Interpretation guide:
#         A ≈ C ≫ B   -> you are largely measuring recording quality. Report honestly, and lean the
#                        conclusions on §3's univariate differences rather than on classification.
#         B ≈ C ≥ A   -> genuine support for H1/H2. The strong result for the report.
#         B ≫ A       -> the best possible outcome: style beats production.
#     Returns : DataFrame of CV scores per feature set
#     Notes   : partially overlaps §6.3 — compute once, present under whichever heading fits the
#               report's argument.

In [ ]:
# ---------------------------------------------------------------------------
# 7.3  Loudness / codec sensitivity
# ---------------------------------------------------------------------------
# def codec_sensitivity_check(manifest, sample_n: int = 20) -> pd.DataFrame
#     Purpose : test whether the format pipeline alone is discriminative.
#     Ideas   : - compare the spectral rolloff / energy above 10 kHz per class. GTZAN originates
#                 from lossy, bandwidth-limited sources and often shows a hard cutoff that iTunes
#                 AAC at 256 kbps does not. A single feature like "energy above 11 kHz" can separate
#                 the classes perfectly for reasons that have nothing to do with music.
#                 -> If you see this, LOW-PASS BOTH CLASSES to a common ceiling, or drop the
#                    high-frequency features, and document it.
#               - re-run §4 with RMS normalisation on vs. off and compare accuracy.
#     Returns : small diagnostic DataFrame + a figure of the mean spectrum per class
#     ⚠️ Run the mean-spectrum-per-class plot EARLY — before §4, ideally right after §2. It takes
#        ten minutes and can save the entire interpretation of your results.

---# 8. Export for the reportKeep every number and figure that appears in the written report reproducible from this notebook.No hand-typed values, no screenshots of DataFrames.

In [ ]:
# ---------------------------------------------------------------------------
# 8.1  Figure saving helper
# ---------------------------------------------------------------------------
# def save_figure(fig, name: str, fmt: str = "pdf", dpi: int = 300) -> Path
#     Purpose : uniform export of every figure to FIG_DIR / f"{name}.{fmt}".
#     Args    : fmt — "pdf" for LaTeX (vector, scales cleanly), "png" for slides
#     Returns : the written path
#     Notes   : set a consistent figure size and font size ONCE at the top of §0 via plt.rcParams,
#               so all figures in the report match. Check that text is readable at print size.

In [ ]:
# ---------------------------------------------------------------------------
# 8.2  Table export
# ---------------------------------------------------------------------------
# def export_table(df, name: str, float_fmt: str = "%.3f") -> tuple[Path, Path]
#     Purpose : write a results table as both .csv (for the repo) and .tex (for the report).
#     Method  : df.to_csv(...) and df.to_latex(..., float_format=float_fmt, index=False)
#     Returns : (csv_path, tex_path)
#
# Tables the report needs:
#   Table 1  dataset summary (n per class, per album, total duration)       <- §1.5
#   Table 2  feature comparison with effect sizes and adjusted p-values     <- §3.3
#   Table 3  model comparison, CV mean ± std                                <- §4.5
#   Table 4  ablation by feature group                                      <- §6.3
#   Table 5  confound checks                                                <- §7

In [ ]:
# ---------------------------------------------------------------------------
# 8.3  One-click reproduction
# ---------------------------------------------------------------------------
# def run_all(force_features: bool = False) -> dict
#     Purpose : call every stage in order and regenerate every figure and table from scratch.
#               Lets you verify before the deadline that the notebook still runs top to bottom.
#     Returns : dict of the main result objects
#     Notes   : run "Restart kernel and run all" at least once, the day BEFORE you submit.
#               Hidden state from out-of-order execution is the classic way a group notebook
#               breaks on the examiner's machine.

---# 9. Results, discussion and limitations*Fill this in as prose while you work — not the night before. Each bullet is a placeholder for aparagraph in the report.*## 9.1 Summary of findings- H1 (rhythmic irregularity): supported / not supported — which features, how large an effect?- H2 (microtonality): supported / not supported — what did `pit_tet_deviation_mean` show?- H3 (classification): best model, grouped-CV F1 (mean ± std), vs. baseline.- Ablation: which feature family carries the signal?## 9.2 Limitations- **Tiny effective sample size.** ~200 clips, but only ~20–25 *independent* musical sources.  All statistics should be read against that number, not against 200.- **One artist defines the positive class.** The model learns "is this Angine de Poitrine",  which is not the same as "is this microtonal / rhythmically irregular music". Any claim about  style in general is an extrapolation — phrase the conclusions accordingly.- **Production confound** (§7): different eras, studios, codecs and mastering.- **GTZAN's known defects**: duplicates, mislabelling, distortion (Sturm 2013), and it is a  1990s–2000s snapshot of "rock", not a definition of it.- **Feature choices are not neutral**: beat trackers and chroma estimators assume 12-TET and a  steady pulse. Applied to music that violates those assumptions, they degrade — and that  degradation is partly what the classifier detects. This is worth a paragraph in itself: it is  the most interesting methodological point in the project.- Segments within a track are not independent → nominal p-values are optimistic.## 9.3 Future work- More microtonal/irregular artists as positives, so the class is a *style* rather than an artist.- A control set matched for era and production (e.g. same-decade rock) to isolate style from mastering.- Symbolic / f0-trajectory analysis rather than frame-level summary statistics.- Listening test: can human listeners classify the same 30 s clips, and where do they fail?## 9.4 Ethics and data handling- The albums were purchased; audio is not redistributed, and only derived features are committed.- GTZAN is used under its research-use terms; cite it and cite Sturm's critique.- The report names the artist; the analysis is critical-descriptive, not evaluative.

---# Appendix## A. EnvironmentOutput of `log_environment()` goes here (§0.3). `requirements.txt` suggestion:```numpypandasscipyscikit-learnlibrosasoundfilematplotlibseabornstatsmodelstqdmjoblibpyarrow          # parquet cache```Pin versions before submission (`pip freeze > requirements.txt`). **ffmpeg** is a separatesystem-level install, needed for §1.1 — note the install command for Windows in the README.## B. References to cite- Tzanetakis & Cook (2002), *Musical genre classification of audio signals* — GTZAN's origin- Sturm (2013), *The GTZAN dataset: its contents, its faults...* — the dataset critique- McFee et al. (2015), *librosa: Audio and music signal analysis in Python*- Pedregosa et al. (2011), *scikit-learn*- Add the course literature on rhythm/pitch features and whatever you use for the tuning-deviation method.## C. Open questions for the group1. Do the two AdP albums differ enough that §7.1 is worth reporting as a main result?2. Shall we add a third, "hard-negative" class (prog/experimental rock) as a stretch goal?   It would make the result far more convincing than AdP vs. mainstream rock.3. Who writes which section of the report, and by when?4. Submission format — notebook plus PDF report, or notebook only? (Check the course page.)